# 5.6 Assumptions of Regression

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.06_assumptions-of-regression.ipynb)

Companion code for Section 5.6 of the book. Run the cells in order from top to bottom.

**Example 5.6.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

# Fit model
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop('MedHouseVal', axis=1)
y = housing.frame['MedHouseVal']
X_s = StandardScaler().fit_transform(X)
model = LinearRegression().fit(X_s, y)
y_pred    = model.predict(X_s)
residuals = y - y_pred

# ── Test 1: Normality (Shapiro-Wilk on sample) ────────────────────
sample_res = residuals.sample(1000, random_state=42)
stat_sw, p_sw = stats.shapiro(sample_res)
print(f'Shapiro-Wilk: statistic={stat_sw:.4f}  p={p_sw:.4e}')
print(f'  Normality: {"OK" if p_sw > 0.05 else "VIOLATED (p<0.05)"}  (large datasets often fail this)')

# ── Test 2: Homoscedasticity (Breusch-Pagan) ──────────────────────
# Regress squared residuals on fitted values
res_squared = residuals**2
aux_model   = LinearRegression().fit(y_pred.values.reshape(-1,1), res_squared)
aux_r2      = aux_model.score(y_pred.values.reshape(-1,1), res_squared)
print(f'\nBreusch-Pagan proxy R²: {aux_r2:.4f}')
print(f'  Homoscedasticity: {"OK" if aux_r2 < 0.05 else "VIOLATED - residual variance is not constant"}')

# ── Test 3: Autocorrelation (Durbin-Watson) ───────────────────────
# DW statistic: 2 = no autocorrelation; <1.5 = positive; >2.5 = negative
diffs = np.diff(residuals.values)
dw = np.sum(diffs**2) / np.sum(residuals.values**2)
print(f'\nDurbin-Watson: {dw:.4f}')
print(f'  Autocorrelation: {"None (OK)" if 1.5 < dw < 2.5 else "POTENTIAL ISSUE"}')

# ── Test 4: Linearity (Residual pattern check) ────────────────────
# Compute correlation between residuals and fitted values (should be ~0)
res_fit_corr = np.corrcoef(y_pred, residuals)[0,1]
print(f'\nResidual-Fitted correlation: {res_fit_corr:.6f}  (should be ≈ 0)')

# ── Diagnostic plots ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Linear Regression Assumption Diagnostics', fontsize=14, fontweight='bold')

# 1. Residuals vs Fitted - Linearity
ax = axes[0,0]
ax.scatter(y_pred, residuals, alpha=0.2, s=5, color='#1976D2')
ax.axhline(0, color='red', lw=2, linestyle='--')
# Add LOWESS smooth to detect pattern
from statsmodels.nonparametric.smoothers_lowess import lowess
smoothed = lowess(residuals, y_pred, frac=0.2, return_sorted=True)
ax.plot(smoothed[:,0], smoothed[:,1], 'orange', lw=2, label='LOWESS trend')
ax.set_xlabel('Fitted Values (ŷ)'); ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted\n(Testing Linearity)'); ax.legend(); ax.grid(alpha=0.3)

# 2. Q-Q Plot - Normality
ax = axes[0,1]
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Normal Q-Q Plot\n(Testing Normality of Residuals)')

# 3. Scale-Location - Homoscedasticity
ax = axes[1,0]
std_residuals = (residuals - residuals.mean()) / residuals.std()
ax.scatter(y_pred, np.sqrt(np.abs(std_residuals)), alpha=0.2, s=5, color='#43A047')
smoothed2 = lowess(np.sqrt(np.abs(std_residuals.values)), y_pred.values, frac=0.2, return_sorted=True)
ax.plot(smoothed2[:,0], smoothed2[:,1], 'orange', lw=2, label='LOWESS trend')
ax.set_xlabel('Fitted Values (ŷ)'); ax.set_ylabel('√|Standardised Residuals|')
ax.set_title('Scale-Location Plot\n(Testing Homoscedasticity)'); ax.legend(); ax.grid(alpha=0.3)

# 4. Residual histogram
ax = axes[1,1]
ax.hist(residuals, bins=60, color='#7B1FA2', alpha=0.7, edgecolor='white', density=True)
# Overlay normal distribution
x_norm = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
       'r-', lw=2, label='Normal distribution')
ax.set_xlabel('Residuals'); ax.set_ylabel('Density')
ax.set_title('Residual Histogram\n(Testing Normality)'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regression_assumptions.png', dpi=150, bbox_inches='tight')
plt.show()